# Experiment: Case-001 Mitapivat Evidence-To-Action Gate

Objective: verify that current mitapivat evidence states map to allowed public actions without turning adult evidence, pediatric watchlist records, or safety text into patient-specific advice.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

evidence_map = json.loads(
    Path(
        "data/regulatory/mitapivat/2026-05-16-mitapivat-evidence-to-action-map.json"
    ).read_text()
)

required_fields = {
    "state_label",
    "source_type",
    "public_fact",
    "allowed_action",
    "blocked_action",
}
missing_fields = {
    state["state_label"]: sorted(required_fields - set(state))
    for state in evidence_map["evidence_states"]
    if required_fields - set(state)
}
missing_fields

## Plan

- Adult label and adult TDT posted results may be benchmarks.
- Pediatric TDT and NTDT records with no posted results must remain watchlist states.
- Case-specific treatment, eligibility, access, or monitoring actions must remain blocked.

In [ ]:
benchmark_states = {
    state["state_label"]
    for state in evidence_map["evidence_states"]
    if "benchmark" in state["allowed_action"].lower()
}
watchlist_states = {
    state["state_label"]
    for state in evidence_map["evidence_states"]
    if "watchlist" in state["allowed_action"].lower()
}
unsafe_allowed_actions = [
    state["state_label"]
    for state in evidence_map["evidence_states"]
    if any(
        term in state["allowed_action"].lower()
        for term in ("dose", "start", "stop", "recommend", "eligible")
    )
]

summary = {
    "gate_label": evidence_map["gate_label"],
    "states_checked": len(evidence_map["evidence_states"]),
    "benchmark_states": sorted(benchmark_states),
    "watchlist_states": sorted(watchlist_states),
    "unsafe_allowed_actions": unsafe_allowed_actions,
    "missing_fields": missing_fields,
}
summary

## Results

- Adult label and adult TDT posted-results states are benchmark states only.
- Pediatric TDT and pediatric NTDT records remain watchlist states only.
- Decision: continue mitapivat as an evidence benchmark; keep case action blocked until qualified clinician review.

In [ ]:
assert summary["gate_label"] == "mitapivat_evidence_to_action_ready"
assert summary["states_checked"] == 6
assert summary["missing_fields"] == {}
assert summary["unsafe_allowed_actions"] == []
assert summary["benchmark_states"] == [
    "adult_tdt_results_posted",
    "adult_us_label_current",
]
assert summary["watchlist_states"] == [
    "pediatric_ntdt_not_yet_recruiting",
    "pediatric_tdt_not_yet_recruiting",
]

decision = {
    "gate_label": summary["gate_label"],
    "public_action": "adult_benchmark_pediatric_watchlist_case_action_blocked",
    "states_checked": summary["states_checked"],
}
decision

## Next steps

- Use the finding note as the public evidence-to-action boundary.
- Ask hematology only whether mitapivat is worth reviewing, not whether to start it.
- Keep raw records and any owner replies outside the public repo.